# Audio-Augmented LLM: Quick Start Guide

This notebook demonstrates the basic workflow:
1. Text generation with Teacher LLM
2. TTS synthesis
3. Emotion embedding extraction
4. Student model training (overview)

In [ ]:
import sys
sys.path.append('../..')

import torch
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display

from audio_augmented_llm.src.tts_pipeline.tts_engine import TTSEngine
from audio_augmented_llm.src.emotion_encoder.emotion_model import EmotionEncoder
from audio_augmented_llm.src.utils.config_loader import load_config

## 1. Environment Check

In [ ]:
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Number of GPUs: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

## 2. Load Configuration

In [ ]:
config = load_config('../configs/experiment_config.yaml')
print("Configuration loaded successfully!")
print(f"Project: {config['project']['name']}")
print(f"Teacher model: {config['teacher']['model_name']}")
print(f"Student model: {config['student']['model_name']}")

## 3. Initialize TTS Engine

In [ ]:
# Initialize TTS
tts = TTSEngine(
    model_type=config['tts']['model_type'],
    model_path=config['tts']['model_path'],
    sample_rate=config['tts']['sample_rate'],
    language=config['tts']['language']
)

print("TTS engine initialized!")

## 4. Generate Sample Audio

In [ ]:
# Sample texts with different emotions
sample_texts = [
    "I'm so happy to help you with this problem!",
    "This situation is really frustrating and concerning.",
    "I'm deeply sorry for your loss."
]

# Generate audio for first sample
audio = tts.synthesize(sample_texts[0], output_path='../outputs/samples/test_happy.wav')

print(f"Generated audio shape: {audio.shape}")
print(f"Duration: {len(audio) / tts.sample_rate:.2f} seconds")

## 5. Visualize Audio Waveform and Spectrogram

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

# Waveform
axes[0].plot(audio)
axes[0].set_title('Waveform')
axes[0].set_xlabel('Sample')
axes[0].set_ylabel('Amplitude')

# Spectrogram
D = librosa.stft(audio)
S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)
librosa.display.specshow(S_db, sr=tts.sample_rate, x_axis='time', y_axis='hz', ax=axes[1])
axes[1].set_title('Spectrogram')

plt.tight_layout()
plt.show()

## 6. Initialize Emotion Encoder

In [ ]:
# Initialize emotion encoder
emotion_encoder = EmotionEncoder(
    model_type=config['emotion_encoder']['model_type'],
    model_name=config['emotion_encoder']['model_name'],
    embedding_dim=config['emotion_encoder']['embedding_dim'],
    num_emotion_classes=config['emotion_encoder']['num_emotion_classes']
)

print(f"Emotion encoder initialized!")
print(f"Model type: {config['emotion_encoder']['model_type']}")
print(f"Embedding dim: {config['emotion_encoder']['embedding_dim']}")

## 7. Extract Emotion Embeddings

In [ ]:
# Extract emotion embedding from generated audio
device = 'cuda' if torch.cuda.is_available() else 'cpu'
emotion_encoder = emotion_encoder.to(device)

emotion_embedding = emotion_encoder.encode_from_file(
    '../outputs/samples/test_happy.wav',
    device=device
)

print(f"Emotion embedding shape: {emotion_embedding.shape}")
print(f"Embedding norm: {torch.norm(emotion_embedding).item():.4f}")

## 8. Visualize Emotion Embeddings

In [ ]:
# Generate embeddings for all sample texts
all_embeddings = []
for i, text in enumerate(sample_texts):
    audio_path = f'../outputs/samples/test_{i}.wav'
    tts.synthesize(text, output_path=audio_path)
    emb = emotion_encoder.encode_from_file(audio_path, device=device)
    all_embeddings.append(emb.cpu().numpy())

# Visualize first 20 dimensions
plt.figure(figsize=(12, 6))
for i, emb in enumerate(all_embeddings):
    plt.plot(emb[0, :20], marker='o', label=f"Sample {i+1}")

plt.xlabel('Embedding Dimension')
plt.ylabel('Value')
plt.title('Emotion Embeddings (First 20 Dimensions)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 9. Next Steps

1. **Prepare Datasets**: Download IEMOCAP, MELD, and dialogue datasets
2. **Train Emotion Encoder**: Fine-tune on emotion speech data
3. **Generate Teacher Data**: Use Teacher LLM + TTS pipeline
4. **Train Student Model**: Multi-modal distillation with text + emotion embeddings
5. **Evaluate**: Run benchmarks and compare baselines

See `scripts/` directory for training and evaluation scripts.